# 05 – Métricas do Lote 4 (L4), por célula

HIS-01, HIS-02, HIS-05, HIS-06, HIS-07, HIS-08, HIS-09, HIS-10, ADD-01 e ADD-02, segundo `spec/metodologia_metricas.md` (secções 7B e 7D), **por célula de grelha** (`cell`) em vez de por país. ADD-01 e ADD-02 são globais por natureza (sem `cell`).

- **Não lê `processed/`**: lê as tabelas do Lote 1 que estão em `metrics/` (`l1_cal_monthly`, `l1_cal_annual`, `l1_meta_month_validity`, `l1_meta_year_validity`, `l1_obs01_raw_counts`, `l1_cal_coverage`; com `STATS_GRID_DEG != GRID_DEG` lê `l1_stats_cal_monthly`/`l1_stats_cal_annual`). Corra antes o notebook `02_metrics_lote1`. A grelha (`GRID_DEG`, `STATS_GRID_DEG`) tem de ser a da execução do Lote 1; se `_runs` disser outra, o notebook pára.
- **Só células ativas**: as estatísticas históricas calculam-se para as células com atividade em pelo menos `MIN_ACTIVE_YEARS` (5) anos válidos (`active_cells`); as outras teriam NA. Ponha `MIN_ACTIVE_YEARS = 1` para incluir todas as células com atividade.
- **Por fatias**: lê `l1_cal_monthly` por faixa de latitude (`lat_band`), divide as células ativas em blocos de `CHUNK_CELLS`, densifica cada bloco (zeros dentro do intervalo de cobertura de cada série) e calcula as 8 métricas por célula (HIS-05/06 e HIS-09 são as mais pesadas). Cada fatia fica em `metrics/_partial_lote4/b<faixa>_c<n>/` e **retoma** se a sessão cair (`FORCE = True` refaz tudo; uma mudança nas tabelas L1 ou nos parâmetros invalida as fatias). HIS-10 é Theil-Sen, O(n²) por célula e série (todos os pares de anos), vetorizado entre grupos com o mesmo número de anos.
- **ADD-01 e ADD-02** escrevem-se uma só vez: ADD-01 soma `n_raw` de todas as células (DuckDB, fora da memória) e ADD-02 lê `l1_cal_coverage`.
- Consolida com DuckDB em pastas Parquet `metrics/l4_*/`, ordenadas por (`ilat`, `ilon`) e particionadas por `series` e `lat_band`. Escreve `_runs` (linha `lote4`: grelha, commit, parâmetros, linhas por tabela) **por último**.
- O código das métricas vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`.

**Antes de correr:** o atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
METRICS = ROOT / 'metrics'
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'metricas-pipeline', REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

assert METRICS.exists(), f"Nao encontro {METRICS}. Corra primeiro o notebook 02 e confirme o atalho 'Sprinklers'."

In [ ]:
# cell: params
FORCE = False               # True ignora as fatias ja calculadas (_partial_lote4) e refaz tudo

# --- grelha e volume (iguais aos do notebook 02) ---
GRID_DEG = 0.01             # grelha das tabelas de base l1_* (tem de ser a do notebook 02)
STATS_GRID_DEG = GRID_DEG   # grelha das estatisticas; com um valor maior le l1_stats_cal_*
MIN_ACTIVE_YEARS = 5        # poda: so celulas com atividade em >= 5 anos validos (1 = todas as ativas)
CHUNK_CELLS = 20000         # celulas por fatia (uma fatia densa tem celulas x series x meses linhas)

# --- referencia historica (valores por defeito da seccao 7B do documento de metodologia) ---
REF_START = None            # primeiro ano da referencia (None = sem limite)
REF_END = None              # ultimo ano da referencia (None = sem limite)
MIN_REF_YEARS = 5           # abaixo disto as estatisticas ficam NA
MAX_UNKNOWN_DAYS = 0        # dias de cobertura desconhecida tolerados num mes/ano valido
MIN_TREND_YEARS = 8         # minimo de anos validos para a tendencia (HIS-10)

In [ ]:
# cell: defs
import datetime, hashlib, json, shutil, subprocess, tempfile, time
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from metrics import common
from metrics.calendar_metrics import active_cells, densify_annual, densify_monthly
from metrics.coverage_meta import add01_active_satellites, add02_zero_days
from metrics.history_cumulative import his07_ytd, his08_yoy, his10_trend
from metrics.history_distribution import (
    his01_hist_median, his02_hist_band, his05_zscore, his06_percentile, his09_months_above,
)
from metrics.store import ParquetSink, default_partitions, prepare_table, stats_source, write_run, write_table

# tabelas por celula (calculadas por fatia) e tabelas globais (escritas uma vez)
SLICE_NAMES = ['l4_his01_hist_median', 'l4_his02_hist_band', 'l4_his05_zscore', 'l4_his06_percentile',
               'l4_his07_ytd', 'l4_his08_yoy', 'l4_his09_months_above', 'l4_his10_trend']
GLOBAL_NAMES = ['l4_add01_active_satellites', 'l4_add02_zero_days']
NAMES = SLICE_NAMES + GLOBAL_NAMES
SORT = {'l4_his01_hist_median': ['series', 'month'], 'l4_his02_hist_band': ['series', 'month'],
        'l4_his05_zscore': ['series', 'year', 'month'], 'l4_his06_percentile': ['series', 'year', 'month'],
        'l4_his07_ytd': ['series', 'year', 'month'], 'l4_his08_yoy': ['series', 'year', 'month'],
        'l4_his09_months_above': ['series', 'year'], 'l4_his10_trend': ['series']}

common._grid_shape(GRID_DEG)
common._grid_shape(STATS_GRID_DEG)
common.coarsen_cell(np.array([0]), GRID_DEG, STATS_GRID_DEG)     # ValueError se nao for multiplo
if CHUNK_CELLS < 1 or MIN_ACTIVE_YEARS < 1:
    raise ValueError('CHUNK_CELLS e MIN_ACTIVE_YEARS tem de ser >= 1')
STATS_SEPARATE = STATS_GRID_DEG != GRID_DEG

SINK = ParquetSink(METRICS)
PART = METRICS / '_partial_lote4'
IN_MONTHLY = stats_source('l1_cal_monthly', GRID_DEG, STATS_GRID_DEG)
IN_ANNUAL = stats_source('l1_cal_annual', GRID_DEG, STATS_GRID_DEG)
SLICE_INPUTS = [IN_MONTHLY, IN_ANNUAL, 'l1_meta_month_validity', 'l1_meta_year_validity']
INPUTS = SLICE_INPUTS + ['l1_obs01_raw_counts', 'l1_cal_coverage']
REF = dict(ref_start=REF_START, ref_end=REF_END, min_ref_years=MIN_REF_YEARS, max_unknown_days=MAX_UNKNOWN_DAYS)
EMPTY_MONTHLY = pd.DataFrame({'cell': np.array([], dtype='int64'), 'year': np.array([], dtype='int64'),
                              'month': np.array([], dtype='int64'), 'series': np.array([], dtype=object),
                              'cell_days': np.array([], dtype='int64')})
EMPTY_ANNUAL = EMPTY_MONTHLY.drop(columns='month')


# --- E/S atomica e DuckDB ---
def atomic_part(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(value, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    temporary.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def duckdb_connect():
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{Path(tempfile.gettempdir()).as_posix()}/duckdb_spill'")
    return con


def scan(name):
    """Fonte SQL (DuckDB) de uma tabela publicada: todos os Parquet da pasta, com as particoes."""
    pattern = (SINK.path(name) / '**' / '*.parquet').as_posix().replace("'", "''")
    return f"read_parquet('{pattern}', hive_partitioning=true, union_by_name=true)"


def query(sql):
    con = duckdb_connect()
    try:
        return con.execute(sql).df()
    finally:
        con.close()


def scalar(name, expression, where=''):
    """Valor de uma agregacao SQL sobre a tabela publicada `name` (0 se for NULL)."""
    value = query(f'SELECT {expression} AS v FROM {scan(name)} {where}')['v'].iloc[0]
    return 0 if pd.isna(value) else value


def distinct(name, column):
    return query(f'SELECT DISTINCT {column} AS v FROM {scan(name)} WHERE {column} IS NOT NULL ORDER BY 1')['v'].tolist()


def tree_signature(name):
    """Assinatura (nome, tamanho, mtime) dos ficheiros de uma tabela: muda se o notebook 02 a refizer."""
    root = SINK.path(name)
    digest = hashlib.sha1()
    for f in sorted(root.rglob('*.parquet')):
        info = f.stat()
        digest.update(f'{f.relative_to(root).as_posix()}|{info.st_size}|{info.st_mtime_ns}\n'.encode())
    return digest.hexdigest()


def plain_series(frame):
    frame = frame.copy()
    if 'series' in frame.columns:
        frame['series'] = frame['series'].astype(object)
    return frame


def read_band(name, band, columns):
    """Uma faixa de latitude de uma tabela L1 (so as colunas pedidas); filtra pelas particoes."""
    frame = pd.read_parquet(SINK.path(name), columns=columns, filters=[('lat_band', '==', int(band))])
    return plain_series(frame)[columns].reset_index(drop=True)


def check_inputs():
    if SINK.exists('_runs'):
        runs = SINK.read('_runs')
        runs = runs[runs['lote'] == 'lote1']
        if len(runs):
            used = json.loads(runs['params'].iloc[0])
            if float(runs['grid_deg'].iloc[0]) != float(GRID_DEG) or (
                    STATS_SEPARATE and float(used.get('stats_grid_deg', GRID_DEG)) != float(STATS_GRID_DEG)):
                raise ValueError(f"o notebook 02 correu com grid_deg={runs['grid_deg'].iloc[0]} e "
                                 f"stats_grid_deg={used.get('stats_grid_deg')}, mas aqui GRID_DEG={GRID_DEG} e "
                                 f'STATS_GRID_DEG={STATS_GRID_DEG}: ajuste os parametros ou refaca o Lote 1')
    missing = [n for n in INPUTS if not SINK.exists(n)]
    if missing:
        raise FileNotFoundError(f'Corra primeiro o notebook 02_metrics_lote1.ipynb: faltam em {METRICS} as tabelas '
                                + ', '.join(missing))


# --- fatias de celulas: parciais com retoma ---
def chunk_signature(band, k, cells):
    signature = dict(schema=1, lote='lote4', band=int(band), chunk=int(k), n_cells=int(len(cells)),
                     cells=hashlib.sha1(np.ascontiguousarray(cells, dtype='int64').tobytes()).hexdigest(),
                     inputs=INPUT_SIGNATURE, grid_deg=GRID_DEG, stats_grid_deg=STATS_GRID_DEG,
                     chunk_cells=CHUNK_CELLS, min_active_years=MIN_ACTIVE_YEARS, ref=REF,
                     min_trend_years=MIN_TREND_YEARS)
    return json.loads(json.dumps(signature, sort_keys=True))


def chunk_done(directory, signature):
    info = read_json(directory / '_done.json')
    return (isinstance(info, dict) and info.get('signature') == signature
            and all((directory / f'{name}.parquet').is_file() for name in SLICE_NAMES))


def process_chunk(band, k, cells, monthly_band, annual_band, signature):
    """Densifica as celulas da fatia, calcula as metricas e guarda-as em PART/b<faixa>_c<k>/."""
    out = PART / f'b{band}_c{k}'
    out.mkdir(parents=True, exist_ok=True)
    marker = out / '_done.json'
    marker.unlink(missing_ok=True)
    t0 = time.time()
    monthly = densify_monthly(monthly_band[monthly_band['cell'].isin(cells)], VALIDITY_M, cells)
    annual = densify_annual(annual_band[annual_band['cell'].isin(cells)], VALIDITY_Y, cells)
    tables = compute(monthly, annual)
    rows = {}
    for name, table in tables.items():
        atomic_part(prepare_table(table, STATS_GRID_DEG, SORT[name]), out / f'{name}.parquet')
        rows[name] = len(table)
    info = dict(signature=signature, band=int(band), chunk=int(k), n_cells=int(len(cells)), rows=rows,
                dense_rows=int(len(monthly)), seconds=round(time.time() - t0, 1),
                cell_days={str(s): int(v) for s, v in monthly.groupby('series')['cell_days'].sum().items()})
    atomic_json(info, marker)
    return info


def consolidate(name, directories):
    """Junta os parciais `name` das fatias numa tabela publicada (pasta Parquet ordenada por
    (ilat, ilon) e particionada por series/lat_band) e devolve o numero de linhas."""
    files = [d / f'{name}.parquet' for d in directories]
    sizes = [pq.ParquetFile(f).metadata.num_rows for f in files]
    non_empty = [f for f, n in zip(files, sizes) if n]
    if not non_empty:       # sem celulas ativas: tabela vazia mas com o esquema certo
        if files:
            schema = pd.read_parquet(files[0]).iloc[:0]
        else:
            schema = prepare_table(compute(densify_monthly(EMPTY_MONTHLY, VALIDITY_M, []),
                                           densify_annual(EMPTY_ANNUAL, VALIDITY_Y, []))[name],
                                   STATS_GRID_DEG, SORT[name])
        SINK.write(name, schema, ())
        return 0
    schema = pd.read_parquet(non_empty[0]).iloc[:0]
    SINK.write_from_parquet(name, non_empty, order_by=['ilat', 'ilon'] + SORT[name],
                            partition_cols=default_partitions(schema))
    return int(sum(sizes))


def git_commit():
    try:
        return subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True,
                              text=True).stdout.strip() or 'unknown'
    except Exception:
        return 'unknown'


def compute(monthly, annual):
    """As 8 metricas por celula de UMA fatia, a partir das tabelas densas (esquema antigo, com `cell`)."""
    return {
        'l4_his01_hist_median': his01_hist_median(monthly, **REF),
        'l4_his02_hist_band': his02_hist_band(monthly, **REF),
        'l4_his05_zscore': his05_zscore(monthly, **REF),
        'l4_his06_percentile': his06_percentile(monthly, **REF),
        'l4_his07_ytd': his07_ytd(monthly, **REF),
        'l4_his08_yoy': his08_yoy(monthly, max_unknown_days=MAX_UNKNOWN_DAYS),
        'l4_his09_months_above': his09_months_above(monthly, **REF),
        'l4_his10_trend': his10_trend(annual, max_unknown_days=MAX_UNKNOWN_DAYS, min_years=MIN_TREND_YEARS),
    }


def global_tables():
    """ADD-01 (satelites ativos por instrumento e mes, somando n_raw de todas as celulas com DuckDB) e
    ADD-02 (dias sem deteccoes globais, a partir de l1_cal_coverage)."""
    summed = query('SELECT instrument, year, month, satellite, sum(n_raw)::BIGINT AS n_raw '
                   f"FROM {scan('l1_obs01_raw_counts')} GROUP BY instrument, year, month, satellite")
    summed.insert(0, 'cell', common.GLOBAL_CELL)
    return {'l4_add01_active_satellites': add01_active_satellites(summed),
            'l4_add02_zero_days': add02_zero_days(SINK.read('l1_cal_coverage'))}

In [ ]:
# cell: run
check_inputs()
PART.mkdir(exist_ok=True)

# validade (partial / n_unknown_days) so das series presentes nas tabelas mensais publicadas
series_present = [str(s) for s in distinct(IN_MONTHLY, 'series')]
VALIDITY_M = plain_series(SINK.read('l1_meta_month_validity'))
VALIDITY_M = VALIDITY_M[VALIDITY_M['series'].isin(series_present)].reset_index(drop=True)
VALIDITY_Y = plain_series(SINK.read('l1_meta_year_validity'))
VALIDITY_Y = VALIDITY_Y[VALIDITY_Y['series'].isin(series_present)].reset_index(drop=True)
INPUT_SIGNATURE = {name: tree_signature(name) for name in SLICE_INPUTS}

chunk_dirs = []
n_active = 0
for band in sorted(int(b) for b in distinct(IN_MONTHLY, 'lat_band')):
    monthly_band = read_band(IN_MONTHLY, band, ['cell', 'year', 'month', 'series', 'cell_days'])
    cells = active_cells(monthly_band, VALIDITY_M, MIN_ACTIVE_YEARS, MAX_UNKNOWN_DAYS)
    n_active += len(cells)
    annual_band = None                      # so se ler se alguma fatia da faixa precisar de ser calculada
    for k, first in enumerate(range(0, len(cells), CHUNK_CELLS)):
        chunk = cells[first:first + CHUNK_CELLS]
        signature = chunk_signature(band, k, chunk)
        directory = PART / f'b{band}_c{k}'
        chunk_dirs.append(directory)
        if not FORCE and chunk_done(directory, signature):
            print(f'[reuse] faixa {band} fatia {k}: {len(chunk)} celulas')
            continue
        if annual_band is None:
            annual_band = read_band(IN_ANNUAL, band, ['cell', 'year', 'series', 'cell_days'])
        info = process_chunk(band, k, chunk, monthly_band, annual_band, signature)
        print(f"[calc] faixa {band} fatia {k}: {info['n_cells']} celulas, {info['dense_rows']} linhas mensais, {info['seconds']} s")
    del monthly_band, annual_band
for stale in PART.iterdir():                # fatias de execucoes anteriores com outra divisao
    if stale.is_dir() and stale not in chunk_dirs:
        shutil.rmtree(stale)
infos = [read_json(d / '_done.json') for d in chunk_dirs]
log = pd.DataFrame([{k: v for k, v in info.items() if k not in ('signature', 'rows', 'cell_days')} for info in infos])
print(f'{n_active} celulas ativas (>= {MIN_ACTIVE_YEARS} anos validos), {len(chunk_dirs)} fatias')

In [ ]:
# cell: publish
published = {}
for name in SLICE_NAMES:
    published[name] = consolidate(name, chunk_dirs)
    print(f'{name}: {published[name]} linhas')

# tabelas globais por natureza: escrevem-se uma vez, sem celula
for name, table in global_tables().items():
    write_table(SINK, name, table, grid_deg=GRID_DEG, partition_by=[])
    published[name] = len(table)
    print(f'{name}: {len(table)} linhas')

# _runs escreve-se POR ULTIMO: uma execucao que parou a meio nao fica registada
write_run(SINK, 'lote4', commit=git_commit(), grid_deg=GRID_DEG, tables=published, params={
    'grid_deg': GRID_DEG, 'stats_grid_deg': STATS_GRID_DEG, 'min_active_years': MIN_ACTIVE_YEARS,
    'chunk_cells': CHUNK_CELLS, 'ref_start': REF_START, 'ref_end': REF_END, 'min_ref_years': MIN_REF_YEARS,
    'max_unknown_days': MAX_UNKNOWN_DAYS, 'min_trend_years': MIN_TREND_YEARS, 'inputs': INPUTS,
    'n_active_cells': n_active, 'n_chunks': len(chunk_dirs),
    'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')})
display(log)

## Verificação
Confirma invariantes entre as tabelas (falha com `AssertionError` se alguma não bater): chaves únicas, uma linha de HIS-05/06/07/08 por linha densa, percentis em [0, 100], `p10 <= p90`, `n_months_above <= n_months_evaluated`, `yoy_pct` só em pares válidos e ADD-01/02 coerentes com `l1_obs01_raw_counts` e `l1_cal_coverage`.

In [ ]:
# cell: verify
KEYS = {'l4_his01_hist_median': ['cell', 'series', 'month'], 'l4_his02_hist_band': ['cell', 'series', 'month'],
        'l4_his05_zscore': ['cell', 'series', 'year', 'month'], 'l4_his06_percentile': ['cell', 'series', 'year', 'month'],
        'l4_his07_ytd': ['cell', 'series', 'year', 'month'], 'l4_his08_yoy': ['cell', 'series', 'year', 'month'],
        'l4_his09_months_above': ['cell', 'series', 'year'], 'l4_his10_trend': ['cell', 'series'],
        'l4_add01_active_satellites': ['instrument', 'year', 'month'], 'l4_add02_zero_days': ['instrument', 'year', 'month']}
for name, keys in KEYS.items():
    n_rows = scalar(name, 'count(*)')
    n_keys = query(f"SELECT count(*) AS v FROM (SELECT DISTINCT {', '.join(keys)} FROM {scan(name)})")['v'].iloc[0]
    assert n_rows == n_keys == published[name], f'{name}: chaves repetidas ou linhas != parciais ({n_rows}/{n_keys})'

dense_rows = sum(info['dense_rows'] for info in infos)
for name in ['l4_his05_zscore', 'l4_his06_percentile', 'l4_his07_ytd', 'l4_his08_yoy']:
    assert published[name] == dense_rows, f'{name}: uma linha por linha de l1_cal_monthly densa ({published[name]} != {dense_rows})'
assert published['l4_his01_hist_median'] == published['l4_his02_hist_band']
assert scalar('l4_his06_percentile', 'count(*)', 'WHERE percentile < 0 OR percentile > 100') == 0, 'percentil fora de [0, 100]'
assert scalar('l4_his02_hist_band', 'count(*)', 'WHERE p10 > p90') == 0, 'p10 > p90'
assert scalar('l4_his09_months_above', 'count(*)', 'WHERE n_months_above > n_months_evaluated') == 0
assert scalar('l4_his08_yoy', 'count(*)', 'WHERE NOT valid_pair AND yoy_pct IS NOT NULL') == 0, 'yoy_pct so existe para pares validos'
assert scalar('l4_add02_zero_days', 'count(*)', 'WHERE pct_zero_days < 0 OR pct_zero_days > 100 OR n_zero_days > n_days') == 0
assert scalar('l4_add01_active_satellites', 'count(*)', 'WHERE n_satellites < 1') == 0
n_obs_months = query(f"SELECT count(*) AS v FROM (SELECT instrument, year, month FROM {scan('l1_obs01_raw_counts')} "
                     'GROUP BY instrument, year, month HAVING sum(n_raw) > 0)')['v'].iloc[0]
assert published['l4_add01_active_satellites'] == n_obs_months, 'ADD-01 != meses com observacoes em OBS-01'
assert scalar('l4_his01_hist_median', 'count(DISTINCT cell)') == n_active, 'celulas de HIS-01 != celulas ativas'
print('OK: invariantes verificadas')
for name in NAMES:
    print(f'{name}: {published[name]} linhas')
print(f'L4 completo: {len(NAMES)} tabelas, {n_active} celulas ativas, escritas em {METRICS}')